In [857]:
# Read it in to inspect it
with open("wonderland.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Length of dataset in characters: ", len(text))

Length of dataset in characters:  144009


In [858]:
tokens = list(map(int, text.encode("utf-8")))
print(len(tokens))

144009


In [859]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    # in the list of ints (ids) replace all consecutive occurences of pair with new token idx
    newids = []
    i = 0
    while i < len(ids):
        # if we are not at the very last position AND the pair matches then replace it
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [860]:
vocab_size = 512
num_merges = vocab_size - 256
ids = list(tokens)

merges = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merging {pair} into new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

merging (101, 32) into new token 256
merging (32, 116) into new token 257
merging (32, 97) into new token 258
merging (257, 104) into new token 259
merging (105, 110) into new token 260
merging (116, 32) into new token 261
merging (101, 114) into new token 262
merging (100, 32) into new token 263
merging (111, 117) into new token 264
merging (32, 115) into new token 265
merging (259, 256) into new token 266
merging (111, 110) into new token 267
merging (44, 32) into new token 268
merging (260, 103) into new token 269
merging (115, 32) into new token 270
merging (116, 104) into new token 271
merging (101, 110) into new token 272
merging (258, 110) into new token 273
merging (10, 10) into new token 274
merging (111, 32) into new token 275
merging (108, 105) into new token 276
merging (101, 97) into new token 277
merging (104, 97) into new token 278
merging (32, 32) into new token 279
merging (104, 256) into new token 280
merging (111, 114) into new token 281
merging (108, 108) into new t

In [861]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 144009
ids length: 67507
compression ratio: 2.13X


In [862]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break # nothing else can be merged
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

In [863]:
test = encode(text)
len(test)
decoded_text = decode(test)
decoded_text == text

True

In [864]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [865]:
# hyperparameters
batch_size = 1  # how many independent sequences will we process in parallel?
block_size = 128 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 1024
n_head = 1
n_layer = 1
dropout = 0.0
# ------------

# torch.manual_seed(1337)

In [866]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:100]) # the first 100 characters

torch.Size([67507]) <built-in method type of Tensor object at 0xffff1e5c5090>
tensor([ 67,  72,  65,  80,  84,  69,  82,  32,  73, 310,  68, 391, 266,  82,
        406,  98, 287,  45,  72, 111, 387, 274, 336, 357, 428, 260, 110, 269,
        305, 388, 261, 309, 121, 257, 363, 308, 284, 265, 287, 116, 297,  98,
        283, 294, 265, 307, 116, 340, 267, 429,  98, 285, 107, 325, 332, 278,
        118, 297, 333, 271, 269, 305, 412,  58,  32, 267,  99, 256, 281, 257,
        327,  99, 256, 355, 398, 112, 367, 112, 308, 260, 435, 429,  98, 314,
        349, 294, 265, 307, 116, 340, 357, 365, 100, 269, 268, 452, 311, 398,
        110, 275])


In [867]:
# split data into training and validation sets
# use the first 90% for training and the remaining 10% for validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [868]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [869]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        # wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [870]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [871]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [872]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [873]:
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters()), 'parameters')

13775360 parameters


In [874]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [875]:
def generate_output():

    prompt = " "
    # prompt = "How are you Alice? "

    context = torch.tensor(
        encode(prompt),
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

In [876]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("**** OUTPUT ****")
generate_output()
print("****.       ****")

step 0: train loss 6.4055, val loss 6.4047
step 100: train loss 5.4058, val loss 5.4210
step 200: train loss 4.8899, val loss 4.9051
step 300: train loss 4.6774, val loss 4.7438
step 400: train loss 4.5028, val loss 4.6225
step 500: train loss 4.3608, val loss 4.5069
step 600: train loss 4.2535, val loss 4.3817
step 700: train loss 4.1268, val loss 4.2747
step 800: train loss 4.1079, val loss 4.2587
step 900: train loss 3.9989, val loss 4.1687
step 1000: train loss 3.9891, val loss 4.1599
step 1100: train loss 3.8857, val loss 4.0838
step 1200: train loss 3.8318, val loss 4.0356
step 1300: train loss 3.8140, val loss 3.9869
step 1400: train loss 3.7909, val loss 4.0372
step 1500: train loss 3.7586, val loss 4.0077
step 1600: train loss 3.7627, val loss 3.9991
step 1700: train loss 3.7249, val loss 3.9387
step 1800: train loss 3.6900, val loss 3.9359
step 1900: train loss 3.6900, val loss 3.9296
step 2000: train loss 3.6557, val loss 3.9313
step 2100: train loss 3.6895, val loss 3.9215
